- Terraform: 
    - It's an IaC (Infrastructure as Code) tool that means instead of manually setting up servers, databases, or networks, we write code to define our infrastructure and terraform creates it for us.
    - With the help of this we can create resource, storage or database, automate deployments
    - Founded in 2014

- Provisioning => create
- Automated provisioning of Infrastructure is called Terraform

- Terraform manages Infrastructure
- Ansible manages configuration of these infrastructure

- Basic Syntax:
    - ```bash
        <block> <resource> <file_name> {
            <arguments>
        }
    ```
- Blocks are of 3 types
    - Resource
    - Variable
    - Output

- example we have to build a aws ec2 instance then we will use resource as a block

- Running a file
    - Firstly use 
        ```bash
            terraform init
        ```
        - By using this now we get the environment to run terraform files
    - To check is the file is valid or not use
        ```bash
            terraform validate
        ```
    - To preview of changes before anything happens use:
        ```bash
            terraform plan
        ```
    - To run the file
        ```bash
            terraform apply
        ```
        - This command create one extra file with extension *.tfstate, it stores terraform's memory or meta-data of our infrastructure
        - and to refresh this state use
        ```bash
            terraform refresh
        ```
    - To destroy what we did use
        ```bash
            terraform destroy
        ```

- For AWS resource we create a provider.tf to select the region, so that our resource will we created
- provider.tf
```bash
    provider "aws"{
        region: "us-east-2"
    }
```
- Interpolation: It's way to inherit or extract the values from a terraform block

- To create a ec2 instance we have to create many things like key pair, vpc, a security group and then our ec2
```bash
# -------------------------------
# Create an AWS Key Pair
# -------------------------------
resource "aws_key_pair" "deployer" {

  # Name of the key pair that will appear in AWS
  key_name = "terra-ec2-kypr"

  # Reads your local public key file and uploads it to AWS
  # This allows SSH access to the EC2 instance using the corresponding private key
  public_key = file("terra-key-ec2.pub") # To create a key type in console ssh-keygen
}


# -------------------------------
# Use Default VPC
# -------------------------------
resource "aws_default_vpc" "default" {

  # This ensures Terraform uses the default VPC in your AWS account
  # If it doesn't exist, Terraform will create one automatically
}


# -------------------------------
# Create Security Group
# -------------------------------
resource "aws_security_group" "terra-security-group" {

  # Name of the security group
  name = "automate-security"

  # Description of what this security group is for
  description = "This is to open the required ports"

  # Attach this security group to the default VPC
  vpc_id = aws_default_vpc.default.id


  # -------- Inbound Rules (Ingress) --------

  ingress {
    # Allow SSH access (port 22)
    from_port = 22
    to_port   = 22
    protocol  = "tcp"

    # Allow access from anywhere (not recommended for production)
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    # Allow HTTP traffic (port 80)
    from_port = 80
    to_port   = 80
    protocol  = "tcp"

    # Open to the public internet
    cidr_blocks = ["0.0.0.0/0"]
  }

  ingress {
    # Allow custom application port (8000)
    from_port = 8000
    to_port   = 8000
    protocol  = "tcp"

    # Open to the public internet
    cidr_blocks = ["0.0.0.0/0"]
  }


  # -------- Outbound Rules (Egress) --------

  egress {
    # Allow all outbound traffic
    from_port = 0
    to_port   = 0

    # "-1" means all protocols (TCP, UDP, ICMP, etc.)
    protocol = "-1"

    # Allow traffic to anywhere
    cidr_blocks = ["0.0.0.0/0"]
  }


  # -------- Tags --------

  tags = {
    # Name tag for easy identification in AWS console
    Name = "automate-sg"
  }
}


# -------------------------------
# Launch EC2 Instance
# -------------------------------
resource "aws_instance" "terraform-ec2" {
  count = n # N defines number of instances

  depends_on = [aws_security_group.terra-security-group] # This defines this resource is depend on what thing
  # Attach the previously created key pair for SSH access
  key_name = aws_key_pair.deployer.key_name

  # Attach the security group by name
  security_groups = [aws_security_group.terra-security-group.name]

  # Instance type (free tier eligible)
  instance_type = "t3.micro"

  # AMI ID (Amazon Machine Image)
  # This defines the OS and pre-installed software
  ami = "ami-07062e2a343acc423"


  # -------- Root Volume Configuration --------

  root_block_device {

    # Size of the root disk in GB
    volume_size = 15

    # Type of EBS volume (gp3 = general purpose SSD)
    volume_type = "gp3"
  }


  # -------- Tags --------

  tags = {
    # Name of the EC2 instance
    Name = "terraform-ec2"
  }
}
```

- Inbound traffic (ingress): Data coming into our server
- Outbound traffic (egress): Data going out from our server

- Now here in above, we are manually hardcoding values that we might change in future, for example aws instance type , so to avoid this we can create a **variable.tf** like
```bash
    variable <variable_name>{
        default=<value>
        type = <variable_type>
    }
```
- And then use in main file as **var.<variable_name>**


- Till now we can not view the output like here in terminal only we don't know the ip's or dns, so for that we can create a output.tf to print these type of things
```bash
    output <output_name>{
        value = resource_type.<resource_name>.<output_request>
    }
```
- For example
```bash
    output "ec2_instance_pulic_ip"{
        value = aws_instance.ec2-instance.public_ip # for single outputs
        value = aws_instance.ec2-instance[*].public_ip # for multiple outputs
    }
```
=================================================================================
- To run this ec2 instance use
```bash
    ssh -i <private_key> ubuntu@<dns>
```
- To reformat all the files use:
```bash 
    terraform fmt
```
- To check all the resources that we created using terraform use
```bash
    terraform state list
```
- And to check for a particular state
```bash
    terraform state show <state_name>
```
- If we want to remove any state management from terraform but not from aws then use
```bash
    terraform state rm <state_name>
```
- And to again add this
```bash
    terraform import <key_name> <key_id>
```

- **for_each** Meta argument:

```bash
    resource "aws_instance" "my-instance"{
        for_each = tomap({ 
            <key>=<value> # Number of key-value pair defines number of instances (meta-argument)
        })
        <type> = each.key/value
    }
```
- For example
```bash
    resource "aws_instance" "my-instance"{
        for_each = tomap({
            app_instance_1 = "t3.micro" # Number of key-value pair defines number of instances,
            app_instance_1 = "t3.micro"
        })
        instance_type = each.value
        tag = {
            Name = each.key
        }
    }
```

- And for output of this for_each is:
```bash
    output "ec2_public_ip"{
        value = [
            for key in aws_instance.my_instance : key.public_ip
        ]
    }
```

1. **Conditional expression:**
- suppose we have a variable as 
```bash
    variable "env"{
        default = "dev"
        type = string
    }
```
- now i want that if this env is prod then my instance will use 20gb other wise 8 gb
```bash
    resource "aws_instance" "my_instance"{
        root_block_device {
            volume_size = var.env == "prod" ? 20 : 8
            volume_type = "gp3"
        }
    }
```

- **Workspace:**
    - It's a way to manage multiple separate states using the same configuration.
    - It's like :
        - One terraform codebase
        - multiple environments (dev, staging, prod) ,like branches concept in github
        - each environment has its own state file 
    - It's like different save slots in a game
- Commands:
    - Show all workspace:
    ```bash
        terraform workspace list
    ```
    - Show current workspace:
    ```bash
        terraform workspace show
    ```
    - Create a new workspace (after using this it will be switched to this state only):
    ```bash
        terraform workspace new <name>
    ```
    - Swtiching workspace:
    ```bash
        terraform workspace select <name>
    ```

- Workspace are not full environment isolation, they just separate state but not separate:
    - Credentials
    - Backend Config
    - Provider Config

- In .tf file we use it like
```bash
    resource "aws_instance" "example" {
        tags = {
            Environment = terraform.workspace
        }
    }
```

- Now, so far for creating instance we are creating many extra things, to avoid this we can use **module** which acts like a template all in one to create an resource, it's like helm in K8s
